In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/bpic20_Rfp.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 147529,2017-02-14 15:34:34,UNKNOWN,organizational unit 65458,137.526306,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 147529,2017-02-14 15:34:43,UNKNOWN,organizational unit 65458,137.526306,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,9.0
2,request for payment 147529,2017-02-15 14:48:02,UNKNOWN,organizational unit 65458,137.526306,Request Payment,SYSTEM,UNDEFINED,83599.0
3,request for payment 147529,2017-02-20 17:32:08,UNKNOWN,organizational unit 65458,137.526306,Payment Handled,SYSTEM,UNDEFINED,441846.0
4,request for payment 147534,2017-03-02 15:55:43,UNKNOWN,organizational unit 65463,59.567024,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
5,request for payment 147534,2017-03-02 15:58:27,UNKNOWN,organizational unit 65463,59.567024,Request For Payment APPROVED by PRE_APPROVER,STAFF MEMBER,PRE_APPROVER,164.0
6,request for payment 147534,2017-03-02 16:07:38,UNKNOWN,organizational unit 65463,59.567024,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,551.0
7,request for payment 147534,2017-03-06 13:57:31,UNKNOWN,organizational unit 65463,59.567024,Request Payment,SYSTEM,UNDEFINED,337793.0
8,request for payment 147534,2017-03-13 17:31:05,UNKNOWN,organizational unit 65463,59.567024,Payment Handled,SYSTEM,UNDEFINED,617614.0
9,request for payment 147539,2017-03-06 14:40:07,UNKNOWN,organizational unit 65458,47.927757,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [3.00, 325445.40]                        57230.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [10.54, 665.70]                          74.7009    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
or

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR'}]

In [13]:
engine.branching_sets

[{'Request For Payment APPROVED by ADMINISTRATION',
  'Request For Payment APPROVED by BUDGET OWNER',
  'Request For Payment APPROVED by PRE_APPROVER',
  'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR',
  'Request For Payment SUBMITTED by EMPLOYEE'},
 {'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR'},
 {'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR'},
 {'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Rfp-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/200 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 176718,4,1,0,0.360782,0.246564,0.475000,0.566667,0.181818,...,0.598926,0.181818,0.083774,0.166667,0.000882,0.333333,0.000000,0.000000,0.0,0.000000
1,0,request for payment 166711,5,1,0,0.325844,0.326688,0.325000,0.427778,0.307692,...,0.307692,0.307692,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
2,0,request for payment 184068,5,1,0,0.368048,0.311096,0.425000,0.466667,0.307692,...,0.613290,0.307692,0.083375,0.166667,0.000084,0.222222,0.000000,0.000000,0.0,0.000000
3,0,request for payment 182365,5,1,0,0.307463,0.289926,0.325000,0.372222,0.307692,...,0.307692,0.307692,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
4,0,request for payment 168709,5,1,0,0.329244,0.250155,0.408333,0.461111,0.307692,...,0.502137,0.307692,0.083333,0.166667,0.000000,0.111111,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,16,request for payment 178121,9,1,0,0.260399,0.249369,0.271429,0.300000,0.561905,...,1.081692,0.142857,0.000107,0.000000,0.000214,0.047619,0.891109,0.891109,1.0,1.000000
131,16,request for payment 170670,9,1,0,0.443311,0.375907,0.510714,0.611905,0.285714,...,0.689889,0.285714,0.118460,0.214286,0.022635,0.285714,0.000000,0.900031,0.0,1.000000
132,16,request for payment 171283,9,1,0,0.444588,0.389175,0.500000,0.602381,0.285714,...,0.631666,0.285714,0.107856,0.214286,0.001427,0.238095,0.000000,0.955376,0.0,1.000000
133,16,request for payment 171342,9,1,0,0.439135,0.360412,0.517857,0.600000,0.285714,...,0.733149,0.285714,0.161720,0.214286,0.109155,0.285714,0.000000,0.735638,0.0,0.999999


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/390 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 176718,4,2,0,0.589974,0.492447,0.6875,0.675000,0.214286,...,0.832460,0.142857,0.250000,0.50,0.000000,0.333333,0.106269,0.586707,0.000000,0.500000
1,0,request for payment 166711,5,2,0,0.480189,0.260378,0.7000,0.600000,0.306250,...,0.541667,0.250000,0.125000,0.25,0.000000,0.166667,0.000000,0.473461,0.000000,0.500000
2,0,request for payment 184068,5,2,0,0.440945,0.381890,0.5000,0.550000,0.318750,...,0.833333,0.250000,0.250000,0.50,0.000000,0.333333,0.000000,0.472625,0.000000,0.500000
3,0,request for payment 182365,5,2,0,0.580394,0.510787,0.6500,0.658333,0.250000,...,0.760639,0.250000,0.125000,0.25,0.000000,0.166667,0.218972,0.692573,0.000000,0.500000
4,0,request for payment 168709,5,2,0,0.597745,0.557989,0.6375,0.716667,0.268750,...,0.927072,0.250000,0.343739,0.25,0.437478,0.333333,0.000000,0.478916,0.000000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,38,request for payment 166918,10,9,0,0.438316,0.426632,0.4500,0.583333,0.624468,...,1.366603,0.000000,0.625000,0.75,0.500000,0.666667,0.074937,0.869956,0.000000,0.888889
242,38,request for payment 170921,10,9,0,0.434743,0.406987,0.4625,0.475000,0.575532,...,1.337842,0.531915,0.125000,0.25,0.000000,0.166667,0.514260,0.851383,0.555555,0.888889
243,38,request for payment 176296,10,9,0,0.532170,0.664340,0.4000,0.541667,0.619149,...,1.047363,0.000000,0.500000,0.50,0.500000,0.500000,0.047363,0.843039,0.000000,0.888889
244,38,request for payment 160855,10,9,0,0.429201,0.208401,0.6500,0.525000,0.560638,...,1.321419,0.000000,0.579816,0.75,0.409632,0.666667,0.074937,0.862026,0.000000,0.888889


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()